In [25]:
!pip install groq python-dotenv

In [26]:
import os
from groq import Groq
from dotenv import load_dotenv
import getpass

load_dotenv()

# Ensure GROQ_API_KEY is set
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Initialize Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [27]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": "You are a highly rigorous technical support expert. You give precise, code-focused, step-by-step debugging help."
    },
    "billing": {
        "system_prompt": "You are an empathetic billing support agent. You explain refund policies, payment issues, and next steps clearly."
    },
    "general": {
        "system_prompt": "You are a friendly general customer support assistant for casual queries."
    },
    "tool": {
        "system_prompt": "You are a tool-using expert. If real-time data is needed, call the appropriate function."
    }
}

In [28]:
def route_prompt(user_input):
    router_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general,tool]

Return ONLY the category name, nothing else.

User query: {user_input}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant", # Updated model name to the latest version
        temperature=0,
        messages=[
            {"role": "system", "content": "You are an intent classification system."},
            {"role": "user", "content": router_prompt}
        ]
    )

    return response.choices[0].message.content.strip().lower()

In [29]:
def process_request(user_input):
    category = route_prompt(user_input)
    if category == "tool":
        # Tool call instead of LLM
        return category, get_bitcoin_price()
    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])["system_prompt"]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return category, response.choices[0].message.content

In [30]:
queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hello, what can you do?"
]

for q in queries:
    category, answer = process_request(q)
    print(f"\nUser Query: {q}")
    print(f"Routed To: {category}")
    print("Response:")
    print(answer)


User Query: My python script is throwing an IndexError on line 5.
Routed To: technical
Response:
To assist you in resolving the issue, I'll need more information about your code. Please provide the following:

1. The exact code snippet where the error is occurring.
2. The full error message, including any stack trace or exception details.
3. A brief description of your code's purpose and any relevant context.

Once I have this information, I can provide a more specific and accurate solution.

In general, `IndexError` in Python usually occurs when you're trying to access an element in a list or other sequence that doesn't exist. Here's a basic example of how to handle this error:

```python
try:
    # Your code here
    my_list = [1, 2, 3]
    print(my_list[3])  # This will raise an IndexError
except IndexError as e:
    print(f"IndexError occurred: {e}")
```

Please provide more details about your code, and I'll be happy to help you debug it.

### Code Review Process

1. You provide y

In [31]:
def get_bitcoin_price():
    # Mock API call (since no internet in Colab sometimes)
    return "The current price of Bitcoin is approximately $62,500 USD (mock data)."

In [32]:
bonus_queries = [
    "What is the current price of Bitcoin?",
    "Tell me today's Bitcoin price",
    "How much is BTC right now?"
]

for q in bonus_queries:
    category, answer = process_request(q)
    print(f"\nUser Query: {q}")
    print(f"Routed To: {category}")
    print("Response:")
    print(answer)


User Query: What is the current price of Bitcoin?
Routed To: billing
Response:
I'm happy to help you with your question, but I need to clarify that I'm a billing support agent, not a cryptocurrency expert. However, I can suggest some options for you to find the current price of Bitcoin.

You can check the current price of Bitcoin on online marketplaces such as Coinbase, Binance, or Kraken. Additionally, you can also check reputable news sources like Bloomberg, CNBC, or The Wall Street Journal for the latest updates on the cryptocurrency market.

If you're experiencing any issues with cryptocurrency transactions or need assistance with a billing query related to cryptocurrency, please let me know and I'll do my best to help.

Is there anything else I can assist you with today?

User Query: Tell me today's Bitcoin price
Routed To: general
Response:
However, I'm a large language model, I don't have real-time access to current market prices. But I can suggest some ways for you to find out